# Goodgorithm — Sentiment CNN training

Trains the small CNN that replaces the VADER placeholder in
`processing/src/pipeline_stages/sentiment.py`. See `CLAUDE.md` in the repo root for full
project context — this is the "ML training" pipeline stage: periodic,
run by a human on a free Colab/Kaggle GPU, not continuous.

**What this notebook does:** loads and harmonizes five public sentiment
datasets into one 3-class (negative/neutral/positive) corpus, trains a
Kim (2014)-style CNN over GloVe-Twitter embeddings, evaluates it, exports
it to ONNX, and uploads it to the `goodgorithm-models` R2 bucket that
`processing/` downloads from at startup.

**Before running:** this notebook fetches
`processing/src/util/sentiment_model.py` from a specific pinned commit on
the repo's `main` branch (not a live "latest" fetch) so the
tokenizer/vocab logic used here exactly matches what `processing/`'s
inference path uses — they can never drift apart even though they run in
totally different environments. If you've changed that file since,
update `SENTIMENT_MODEL_COMMIT` below to the new commit SHA.

In [ ]:
!pip install -q datasets gensim onnx onnxruntime boto3

import random

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)


## Fetch the shared tokenizer/vocab file

Pinned to a commit SHA, not `main` — a later edit to this file shouldn't
silently invalidate a model that's already been published.


In [ ]:
import urllib.request

SENTIMENT_MODEL_COMMIT = "53d2ecab1ba26ebf05321b2bb211c656e238660b"
SENTIMENT_MODEL_RAW_URL = (
    f"https://raw.githubusercontent.com/goodgorithm/goodgorithm/{SENTIMENT_MODEL_COMMIT}"
    f"/processing/src/util/sentiment_model.py"
)

urllib.request.urlretrieve(SENTIMENT_MODEL_RAW_URL, "sentiment_model.py")
import sentiment_model

print("EMBEDDING_DIM", sentiment_model.EMBEDDING_DIM)
print("FILTER_SIZES", sentiment_model.FILTER_SIZES)
print("NUM_FILTERS", sentiment_model.NUM_FILTERS)
print("MAX_SEQ_LEN", sentiment_model.MAX_SEQ_LEN)
print("MAX_VOCAB_SIZE", sentiment_model.MAX_VOCAB_SIZE)
print(sentiment_model.tokenize("Check this out https://example.com/x @friend #blessed :) <3"))

## HuggingFace Hub authentication (optional, avoids rate-limit failures)

Every `load_dataset()` call below hits the HF Hub anonymously by default,
sharing a low, queue-based rate limit across every anonymous caller at
once -- easily hit as a `429 Too Many Requests: maximum queue size
reached` partway through a normal run, not just under unusual load.
Setting `HF_TOKEN` (Colab/Kaggle Secrets, same lookup pattern as the R2
credentials near the end of this notebook) raises that limit by a lot and
is a one-time, free sign-up at
[huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
(a "read" token is enough). Falls back to fully anonymous access if no
token is found -- the notebook still runs without one, just at real risk
of hitting the rate limit.


In [ ]:
import os

HF_TOKEN = None
try:
    from google.colab import userdata

    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass

if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient

        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF_TOKEN set -- dataset loads below use authenticated Hub requests")
else:
    print("No HF_TOKEN found -- proceeding anonymously (see markdown above)")


## Load + harmonize five datasets

All five get mapped to the same 3-class scheme: `0=negative, 1=neutral,
2=positive` (matches TweetEval's native convention, and is the label
order `processing/src/pipeline_stages/sentiment.py` assumes when it computes
`P(positive) - P(negative)` from the model's output).

### Sentiment140

~1.6M tweets, weak-labeled via emoticons (which are stripped from the
text itself). Historically the released train split is binary-only
(labels 0/4) even though the label field's nominal range is 0/2/4 — the
code below doesn't assume this, it just maps whatever values are actually
present.


In [ ]:
from datasets import load_dataset

# stanfordnlp/sentiment140's canonical repo uses a legacy loading script,
# which current `datasets` versions no longer execute ("Dataset scripts
# are no longer supported"). The auto-converted parquet revision works
# without a script and has identical content -- verified directly against
# the real dataset before this notebook was written.
sentiment140 = load_dataset("stanfordnlp/sentiment140", revision="refs/convert/parquet")
print(sentiment140)

SENTIMENT140_LABEL_MAP = {0: 0, 2: 1, 4: 2}  # -> negative, neutral, positive

print("label values present in train:", sorted(set(sentiment140["train"]["sentiment"])))

s140_texts = list(sentiment140["train"]["text"])
s140_labels = [SENTIMENT140_LABEL_MAP[v] for v in sentiment140["train"]["sentiment"]]
print(f"sentiment140: {len(s140_texts)} examples")


### TweetEval / SemEval-2017 Task 4 (sentiment config)

Human-annotated, natively 3-class, best domain + label-quality match to
our own short social-post text. `train` + `validation` go into the
combined training pool; the dataset's own `test` split is held out
entirely and used later as a second, domain-matched evaluation number —
not mixed into our train/val/test split, to avoid any leakage.


In [ ]:
tweet_eval = load_dataset("cardiffnlp/tweet_eval", "sentiment")
print(tweet_eval)
# label convention: 0=negative, 1=neutral, 2=positive — matches our target scheme directly

te_train_texts = list(tweet_eval["train"]["text"]) + list(tweet_eval["validation"]["text"])
te_train_labels = list(tweet_eval["train"]["label"]) + list(tweet_eval["validation"]["label"])
te_test_texts = list(tweet_eval["test"]["text"])
te_test_labels = list(tweet_eval["test"]["label"])
print(f"tweet_eval train+val pool: {len(te_train_texts)}, reserved test: {len(te_test_texts)}")


### Tweet Sentiment Extraction (Kaggle 2020)

Human-annotated, natively 3-class with a real neutral, tweet-length text
— the same profile as TweetEval and a direct counterweight to
Sentiment140's weak emoticon labels dominating the pool. `train` joins
the combined training pool; the dataset's own `test` split is held out
as a second independent eval number alongside TweetEval's, never mixed
into our train/val/test split. Text vintage skews older (2009–2015-era
tweets, like Sentiment140's) — the win here is label quality, not
recency; the per-dataset eval breakdown further down is what tells us
whether that older text still helps or hurts on the domain-matched
benchmarks.


In [ ]:
# mteb/tweet_sentiment_extraction -- the 2020 Kaggle "Tweet Sentiment
# Extraction" competition corpus, re-annotated by hand into 3 classes
# (0=negative, 1=neutral, 2=positive -- already our target scheme). Same
# short-tweet register as TweetEval, ~1.6x its size, and human-labeled
# with a real neutral class rather than Sentiment140's emoticon heuristic.
# train -> combined pool; the dataset's own test split is held out as a
# second independent, never-trained-on eval, exactly like TweetEval's.
tweet_sent_ext = load_dataset("mteb/tweet_sentiment_extraction")
print(tweet_sent_ext)


def clean_tse(split):
    texts, labels = [], []
    for ex in split:
        text = (ex["text"] or "").strip()
        if not text:
            continue  # a handful of rows have empty text
        texts.append(text)
        labels.append(ex["label"])
    return texts, labels


tse_train_texts, tse_train_labels = clean_tse(tweet_sent_ext["train"])
tse_test_texts, tse_test_labels = clean_tse(tweet_sent_ext["test"])
print(
    f"tweet_sentiment_extraction train pool: {len(tse_train_texts)}, "
    f"reserved test: {len(tse_test_texts)}"
)
print(
    "train class counts (0=neg,1=neu,2=pos):",
    {c: tse_train_labels.count(c) for c in (0, 1, 2)},
)


### GoEmotions

58k Reddit comments, 27 fine-grained emotion labels (multi-label) + a
`neutral` class. We only keep single-label examples (multi-label ones are
genuinely ambiguous about overall valence, better excluded than guessed
at), then map through the **official Demszky et al. (2020) sentiment
grouping** — fetched here from the primary source
(`google-research/google-research`), not hand-typed from memory, since
getting this mapping wrong would quietly mislabel a third of the training
data. The paper's own grouping has 12 positive / 11 negative / 4
"ambiguous" categories + neutral; we drop the ambiguous-grouped examples
rather than force them into positive/negative/neutral.


In [ ]:
import urllib.request
import json as jsonlib

SENTIMENT_MAPPING_URL = (
    "https://raw.githubusercontent.com/google-research/google-research/master"
    "/goemotions/data/sentiment_mapping.json"
)
with urllib.request.urlopen(SENTIMENT_MAPPING_URL) as f:
    ge_sentiment_groups = jsonlib.load(f)

print(ge_sentiment_groups)
assert len(ge_sentiment_groups["positive"]) == 12
assert len(ge_sentiment_groups["negative"]) == 11
assert len(ge_sentiment_groups["ambiguous"]) == 4

GE_LABEL_TO_CLASS = {}
for emotion in ge_sentiment_groups["positive"]:
    GE_LABEL_TO_CLASS[emotion] = 2
for emotion in ge_sentiment_groups["negative"]:
    GE_LABEL_TO_CLASS[emotion] = 0
GE_LABEL_TO_CLASS["neutral"] = 1
# "ambiguous"-grouped emotions and multi-label examples are dropped below, not mapped.


In [ ]:
go_emotions = load_dataset("google-research-datasets/go_emotions", "simplified")
print(go_emotions)

ge_names = go_emotions["train"].features["labels"].feature.names


def harmonize_go_emotions(split):
    texts, labels = [], []
    for example in split:
        if len(example["labels"]) != 1:
            continue  # drop multi-label examples — ambiguous overall valence
        emotion = ge_names[example["labels"][0]]
        if emotion not in GE_LABEL_TO_CLASS:
            continue  # dropped "ambiguous"-grouped emotion
        texts.append(example["text"])
        labels.append(GE_LABEL_TO_CLASS[emotion])
    return texts, labels


ge_texts, ge_labels = harmonize_go_emotions(go_emotions["train"])
ge_val_texts, ge_val_labels = harmonize_go_emotions(go_emotions["validation"])
ge_texts += ge_val_texts
ge_labels += ge_val_labels
print(f"go_emotions (single-label, non-ambiguous): {len(ge_texts)} examples")


### dair-ai/emotion

~20k tweets, 6 emotion labels, no native neutral. Contributes only the
polar classes, mapped by the same valence grouping as GoEmotions
(joy/love → positive; sadness/anger/fear → negative; surprise dropped as
ambiguous). `train`+`validation` fold into the pool; the test split is
unused, same as GoEmotions'.


In [ ]:
# dair-ai/emotion -- ~20k tweets labeled with 6 basic emotions. No native
# neutral class, so this only contributes negative/positive, mapped
# through the same valence grouping used for GoEmotions above: joy/love ->
# positive, sadness/anger/fear -> negative, surprise dropped (ambiguous
# valence). Reinforces the two polar classes with clean tweet-domain
# labels without touching the neutral class.
emotion_ds = load_dataset("dair-ai/emotion", "split")
print(emotion_ds)

emo_names = emotion_ds["train"].features["label"].names
assert emo_names == ["sadness", "joy", "love", "anger", "fear", "surprise"], emo_names

EMO_LABEL_TO_CLASS = {
    "joy": 2,
    "love": 2,
    "sadness": 0,
    "anger": 0,
    "fear": 0,
    # "surprise" deliberately absent -> dropped, like GoEmotions' "ambiguous" group
}


def harmonize_emotion(split):
    texts, labels = [], []
    for ex in split:
        emotion = emo_names[ex["label"]]
        if emotion not in EMO_LABEL_TO_CLASS:
            continue
        text = (ex["text"] or "").strip()
        if not text:
            continue
        texts.append(text)
        labels.append(EMO_LABEL_TO_CLASS[emotion])
    return texts, labels


emo_train_texts, emo_train_labels = harmonize_emotion(emotion_ds["train"])
emo_val_texts, emo_val_labels = harmonize_emotion(emotion_ds["validation"])
emo_train_texts += emo_val_texts
emo_train_labels += emo_val_labels
print(f"dair-ai/emotion (surprise dropped): {len(emo_train_texts)} examples")
print("class counts (0=neg,2=pos):", {c: emo_train_labels.count(c) for c in (0, 2)})


## Combine + balance

Sentiment140 (1.6M) would otherwise drown out the four human-labeled
datasets combined — TweetEval, Tweet Sentiment Extraction, GoEmotions,
and dair-ai/emotion, together roughly 150k — and the model would just
learn Sentiment140's weak, emoticon-derived, binary-only signal, losing
both the neutral class and the better-labeled data's influence.

v4 controlled this by discarding Sentiment140 rows down to a fixed
multiple of the other four sources' combined size
(`SENTIMENT140_MAX_MULTIPLE`, `1` was v4's value). That's a blunt
instrument: it fixes Sentiment140's *influence* only by also fixing how
much of it the model ever sees, and it happens to fix the neutral-prior
imbalance as a side effect rather than by design (issue #76's root-cause
finding, issue #171 direction 2).

**`SENTIMENT140_EFFECTIVE_MULTIPLE`** decouples the two. Every Sentiment140
row is kept — nothing is discarded — and each gets a per-example weight so
its *total* weighted contribution to the training loss equals
`SENTIMENT140_EFFECTIVE_MULTIPLE` times the other four sources' combined
size: the same operating point the old cap reached by discarding rows,
now reachable at any multiple without ever throwing data away. That
per-example weight is `f(source) × g(class)`:

- `f(source)` is that multiplier for Sentiment140 rows, `1.0` for every
  other source (all four stay at full, unweighted volume). At
  `SENTIMENT140_EFFECTIVE_MULTIPLE = 1` this is designed to land close to
  v4's numbers — the point isn't to move the operating point, it's to
  test whether the rows the old cap discarded actually carried anything
  useful once they're down-weighted instead of dropped.
- `g(class)` is the standard `"balanced"` formula (`total / (n_classes ×
  class_count)`), but computed over each class's *source-weighted* count,
  not its raw row count — computing it on raw counts would see
  Sentiment140's full, unweighted volume and badly overcorrect for
  neutral before `f(source)` ever gets a chance to act.

`1` below is v4's value, restated as a weighting target instead of a hard
cap — sweep it the same way `SENTIMENT140_MAX_MULTIPLE` was swept in #76,
comparing `best_val_macro_f1`, the TweetEval/TSE reserved-test macro-F1,
and both reserved-test negative-recall rows against v4's fixed baseline
(recorded in #171).

The per-dataset held-out breakdown in the eval section is the companion
diagnostic, same as before.


In [ ]:
from sklearn.model_selection import train_test_split

SENTIMENT140_EFFECTIVE_MULTIPLE = 1  # experiment knob -- see the markdown above

other_pool_size = (
    len(te_train_texts) + len(tse_train_texts) + len(ge_texts) + len(emo_train_texts)
)

# Unlike v4, no rows are discarded here -- Sentiment140 keeps its full
# volume, and its per-example influence is controlled by weighting below.
all_texts = s140_texts + te_train_texts + tse_train_texts + ge_texts + emo_train_texts
all_labels = s140_labels + te_train_labels + tse_train_labels + ge_labels + emo_train_labels
# Per-source tags, carried through both splits so the eval section can
# break held-out performance down by originating dataset (issue #76), and
# so the weighting step below knows which rows are Sentiment140's.
all_sources = (
    ["sentiment140"] * len(s140_texts)
    + ["tweet_eval"] * len(te_train_texts)
    + ["tweet_sentiment_extraction"] * len(tse_train_texts)
    + ["go_emotions"] * len(ge_texts)
    + ["dair_emotion"] * len(emo_train_texts)
)
assert len(all_texts) == len(all_labels) == len(all_sources)

print(f"combined pool: {len(all_texts)} examples (full Sentiment140 volume kept)")
print("by source:", {s: all_sources.count(s) for s in sorted(set(all_sources))})
print(
    "class counts (0=neg,1=neu,2=pos):",
    {c: all_labels.count(c) for c in (0, 1, 2)},
)

train_texts, temp_texts, train_labels, temp_labels, train_sources, temp_sources = train_test_split(
    all_texts, all_labels, all_sources, test_size=0.2, random_state=SEED, stratify=all_labels
)
val_texts, test_texts, val_labels, test_labels, val_sources, test_sources = train_test_split(
    temp_texts, temp_labels, temp_sources, test_size=0.5, random_state=SEED, stratify=temp_labels
)
print(f"train={len(train_texts)} val={len(val_texts)} test={len(test_texts)}")

# f(source): Sentiment140 rows are down-weighted so their *total* weighted
# count matches SENTIMENT140_EFFECTIVE_MULTIPLE x the other four sources'
# combined size; every other source keeps weight 1.0 (full, unweighted
# volume, unchanged from v4).
train_s140_count = train_sources.count("sentiment140")
s140_source_weight = (
    SENTIMENT140_EFFECTIVE_MULTIPLE * (len(train_texts) - train_s140_count) / train_s140_count
    if train_s140_count
    else 1.0
)
source_weight = {
    "sentiment140": s140_source_weight,
    "tweet_eval": 1.0,
    "tweet_sentiment_extraction": 1.0,
    "go_emotions": 1.0,
    "dair_emotion": 1.0,
}
print("per-source weight f(source):", source_weight)

# g(class): the standard "balanced" formula, applied to each class's
# source-weighted count rather than its raw row count -- see the markdown
# above for why raw counts would badly overcorrect for neutral here.
weighted_class_totals = {0: 0.0, 1: 0.0, 2: 0.0}
for label, source in zip(train_labels, train_sources):
    weighted_class_totals[label] += source_weight[source]
total_weighted = sum(weighted_class_totals.values())
class_weight_by_label = {
    c: (total_weighted / (3 * weighted_class_totals[c])) if weighted_class_totals[c] else 0.0
    for c in (0, 1, 2)
}
print("weighted class counts:", weighted_class_totals)
print("g(class):", class_weight_by_label)

train_sample_weights = [
    source_weight[source] * class_weight_by_label[label]
    for source, label in zip(train_sources, train_labels)
]


## Tokenize + build vocabulary

Uses `sentiment_model.tokenize()` — the exact same function
`processing/`'s inference path uses. Vocab is capped at `MAX_VOCAB_SIZE`
by frequency; `<pad>`/`<unk>` are reserved ids 0/1.


In [ ]:
from collections import Counter

train_tokens = [sentiment_model.tokenize(t) for t in train_texts]

token_counts = Counter()
for tokens in train_tokens:
    token_counts.update(tokens)

MIN_FREQ = 2
vocab_words = [w for w, c in token_counts.most_common() if c >= MIN_FREQ]
vocab_words = vocab_words[: sentiment_model.MAX_VOCAB_SIZE - 2]  # room for pad/unk

vocab = {sentiment_model.PAD_TOKEN: 0, sentiment_model.UNK_TOKEN: 1}
for i, w in enumerate(vocab_words, start=2):
    vocab[w] = i

print(f"vocab size: {len(vocab)}")

lengths = [len(t) for t in train_tokens]
p50, p90, p95, p99 = np.percentile(lengths, [50, 90, 95, 99])
print(f"token length percentiles: p50={p50:.0f} p90={p90:.0f} p95={p95:.0f} p99={p99:.0f}")
print(f"MAX_SEQ_LEN={sentiment_model.MAX_SEQ_LEN} -- if p95 exceeds this, consider raising it")


## Embeddings

`EMBEDDING_SOURCE` selects a pretrained embedding to initialize the
lookup table from; rows are fine-tuned during training (not frozen),
which generally outperforms a frozen embedding for this kind of small
classifier. `sentiment_model.SentimentCNN` infers its embedding
dimension from the loaded matrix's own shape (see the Model section
below), so this notebook can freely try a different source or
dimension without touching the shared `sentiment_model.py` file.

Candidates under investigation (issue #171 -- `glove-twitter-100` was
the original pick, on a "trained on tweets" argument that was never
actually tested against alternatives):

- **`glove-twitter-100`** (current default) / **`glove-twitter-200`** --
  same 2013-14 Twitter corpus, only dimensionality differs. Isolates
  "does more capacity per word help" from any domain question.
- **`glove-wiki-gigaword-100`** / **`-300`** -- Wikipedia 2014 + Gigaword
  newswire, formal register, much larger and cleaner vocabulary but
  essentially no social-media-specific vocabulary. Useful mainly as a
  domain-mismatch comparison point.
- **`fasttext-wiki-news-subwords-300`** -- a different training
  algorithm (subword/character-n-gram-based) on a smaller, more formal
  corpus (Wikipedia 2017 + UMBC + statmt.org news) than Twitter/GloVe.
  Loaded here as gensim's precomputed `KeyedVectors` for these word
  vectors, which does **not** retain fastText's subword model -- no
  out-of-vocabulary synthesis, just a different corpus/algorithm's fixed
  vocabulary, same coverage limitation as GloVe for words outside it.
  True OOV synthesis needs the full `cc.en.300.bin` model (~7GB,
  Facebook's own download, loaded via
  `gensim.models.fasttext.load_facebook_vectors` instead of
  `gensim.downloader`) -- a separate, heavier follow-up, not wired up
  here.

Coverage % is the primary comparison metric across sources -- a sharp
drop signals a tokenization/vocab mismatch or (for a cased source) a
casing mismatch against our all-lowercase tokens, not necessarily a
worse embedding.


In [ ]:
import gensim.downloader as gensim_api

EMBEDDING_SOURCE = "glove-twitter-100"  # swap to try a different source -- see markdown above

embedding_kv = gensim_api.load(EMBEDDING_SOURCE)
EMBEDDING_DIM_ACTUAL = embedding_kv.vector_size
print(f"loaded {EMBEDDING_SOURCE}: {len(embedding_kv):,} words, {EMBEDDING_DIM_ACTUAL} dims")

embedding_matrix = np.random.normal(0, 0.1, size=(len(vocab), EMBEDDING_DIM_ACTUAL)).astype(np.float32)
embedding_matrix[vocab[sentiment_model.PAD_TOKEN]] = np.zeros(EMBEDDING_DIM_ACTUAL, dtype=np.float32)

found = 0
for word, idx in vocab.items():
    if word in embedding_kv:
        embedding_matrix[idx] = embedding_kv[word]
        found += 1

print(f"{EMBEDDING_SOURCE} coverage: {found}/{len(vocab)} ({found / len(vocab):.1%})")


## Model

Kim (2014)-style CNN: parallel 1D convolutions over the embedded sequence
at a few different window widths, global max-pool each, concatenate,
dropout, linear classifier. Built from `sentiment_model`'s constants so
the recorded config and the actual model can never disagree.

Hyperparameters live in `sentiment_model.py` (the shared file), not
hardcoded twice here.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class SentimentCNN(nn.Module):
    def __init__(self, embedding_matrix, filter_sizes, num_filters, dropout):
        super().__init__()
        vocab_size, embedding_dim = embedding_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix), freeze=False, padding_idx=0
        )
        self.convs = nn.ModuleList(
            [nn.Conv1d(embedding_dim, num_filters, kernel_size=k) for k in filter_sizes]
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(filter_sizes), 3)

    def forward(self, input_ids):
        x = self.embedding(input_ids)          # (batch, seq_len, emb_dim)
        x = x.transpose(1, 2)                    # (batch, emb_dim, seq_len)
        pooled = []
        for conv in self.convs:
            c = F.relu(conv(x))                   # (batch, num_filters, L')
            pooled.append(F.max_pool1d(c, c.shape[2]).squeeze(2))
        x = torch.cat(pooled, dim=1)                # (batch, num_filters * len(filter_sizes))
        x = self.dropout(x)
        return self.fc(x)                             # (batch, 3) logits


model = SentimentCNN(
    embedding_matrix,
    sentiment_model.FILTER_SIZES,
    sentiment_model.NUM_FILTERS,
    sentiment_model.DROPOUT,
).to(DEVICE)

num_params = sum(p.numel() for p in model.parameters())
print(f"total params: {num_params:,}")


## Train

Adam, early stopping on validation macro-F1 (patience=2).

Loss is a per-example weighted cross-entropy — `nn.CrossEntropyLoss(reduction="none")`,
reduced as a weighted mean using the `f(source) × g(class)` weights built
above — rather than the single class-weight tensor `nn.CrossEntropyLoss(weight=...)`
took before direction 2. The weighted-mean reduction matches how
`CrossEntropyLoss`'s own `weight=` argument normalizes: by the sum of
weights actually seen in the batch, not by batch size.


In [ ]:
from torch.utils.data import DataLoader, Dataset


class SentimentDataset(Dataset):
    def __init__(self, texts, labels, vocab, weights=None):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        # weights unused outside training -- val/test pass no weights, and
        # get an implicit 1.0 so every loader yields the same tuple shape.
        self.weights = weights if weights is not None else [1.0] * len(texts)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids = sentiment_model.encode(sentiment_model.tokenize(self.texts[idx]), self.vocab)
        return (
            torch.tensor(ids, dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long),
            torch.tensor(self.weights[idx], dtype=torch.float32),
        )


# 256 was fine when every source's per-example weight came only from
# class_weight="balanced" (a narrow 2-3x spread). Direction 2's f(source)
# now spans a much wider range within the same batch -- a down-weighted
# Sentiment140 row and a full-weight neutral row from a small source can
# differ ~20x -- so a small batch's weighted mean is a noisier estimate of
# the true population-weighted loss than v4 ever had to deal with. Bumped
# to reduce that per-batch variance; not yet confirmed this is sufficient
# on its own -- see the direction-2 tracking notes in #171.
BATCH_SIZE = 1024
train_loader = DataLoader(
    SentimentDataset(train_texts, train_labels, vocab, train_sample_weights),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
val_loader = DataLoader(SentimentDataset(val_texts, val_labels, vocab), batch_size=BATCH_SIZE)
test_loader = DataLoader(SentimentDataset(test_texts, test_labels, vocab), batch_size=BATCH_SIZE)


In [ ]:
from sklearn.metrics import f1_score

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(reduction="none")

MAX_EPOCHS = 10
PATIENCE = 2

best_val_f1 = -1.0
best_state = None
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for batch_ids, batch_labels, batch_weights in train_loader:
        batch_ids = batch_ids.to(DEVICE)
        batch_labels = batch_labels.to(DEVICE)
        batch_weights = batch_weights.to(DEVICE)
        optimizer.zero_grad()
        logits = model(batch_ids)
        per_example_loss = criterion(logits, batch_labels)
        loss = (per_example_loss * batch_weights).sum() / batch_weights.sum()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch_ids.size(0)

    model.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for batch_ids, batch_labels, _ in val_loader:
            batch_ids = batch_ids.to(DEVICE)
            logits = model(batch_ids)
            val_preds.extend(logits.argmax(dim=1).cpu().tolist())
            val_true.extend(batch_labels.tolist())
    val_f1 = f1_score(val_true, val_preds, average="macro")

    print(f"epoch {epoch}: train_loss={total_loss / len(train_texts):.4f} val_macro_f1={val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"early stopping (no val improvement for {PATIENCE} epochs)")
            break

model.load_state_dict(best_state)
print(f"best val macro-F1: {best_val_f1:.4f}")


## Evaluate

Three views, in order:

1. **Held-out test split from our combined pool** — overall, then broken
   down per originating dataset. The per-dataset breakdown is the issue
   #76 diagnostic: it shows whether Sentiment140 rows score well while
   the domain-matched datasets lag, or whether the gap is uniform.
2. **TweetEval's own reserved test split** — human-labeled,
   domain-matched, never trained on.
3. **Tweet Sentiment Extraction's own reserved test split** — a second
   independent human-labeled benchmark, same role as TweetEval's.

Judge a sweep run mainly on 2 and 3, and on the negative-recall row of
their confusion matrices.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix


def evaluate(texts, labels, vocab, model, label_name):
    loader = DataLoader(SentimentDataset(texts, labels, vocab), batch_size=BATCH_SIZE)
    model.eval()
    preds, true = [], []
    with torch.no_grad():
        for batch_ids, batch_labels, _ in loader:
            batch_ids = batch_ids.to(DEVICE)
            logits = model(batch_ids)
            preds.extend(logits.argmax(dim=1).cpu().tolist())
            true.extend(batch_labels.tolist())

    print(f"--- {label_name} ---")
    print(classification_report(true, preds, target_names=["negative", "neutral", "positive"]))
    print("confusion matrix (rows=true, cols=pred):")
    print(confusion_matrix(true, preds))
    return preds, true


def evaluate_by_source(texts, labels, sources, vocab, model):
    by_source = {}
    for text, label, source in zip(texts, labels, sources):
        bucket = by_source.setdefault(source, ([], []))
        bucket[0].append(text)
        bucket[1].append(label)
    for source in sorted(by_source):
        s_texts, s_labels = by_source[source]
        evaluate(s_texts, s_labels, vocab, model, f"held-out — {source} only (n={len(s_texts)})")


_ = evaluate(test_texts, test_labels, vocab, model, "held-out test split (combined pool)")
evaluate_by_source(test_texts, test_labels, test_sources, vocab, model)
_ = evaluate(te_test_texts, te_test_labels, vocab, model, "TweetEval's own reserved test split")
_ = evaluate(
    tse_test_texts,
    tse_test_labels,
    vocab,
    model,
    "Tweet Sentiment Extraction's own reserved test split",
)


In [ ]:
# Spot-check against a handful of examples. Replace these with real
# examples copied from Supabase `staging.raw_posts` for a domain-specific
# eyeball check -- this notebook has no DB access of its own.
spot_check_examples = [
    "Just adopted the sweetest rescue dog, he already knows how to sit!",
    "Traffic was awful this morning and I spilled coffee on my shirt.",
    "The city council meeting is scheduled for 3pm on Tuesday.",
    "Wow. Just wow. That is the kind of exceptionalist twaddle I'd expect.",
]

model.eval()
with torch.no_grad():
    for text in spot_check_examples:
        ids = torch.tensor(
            [sentiment_model.encode(sentiment_model.tokenize(text), vocab)], dtype=torch.long
        ).to(DEVICE)
        probs = F.softmax(model(ids), dim=1)[0].cpu().numpy()
        score = float(probs[2] - probs[0])
        print(f"{score:+.3f}  neg={probs[0]:.2f} neu={probs[1]:.2f} pos={probs[2]:.2f}  :: {text!r}")


## Export to ONNX

`processing/`'s inference path only needs `onnxruntime`, not full
PyTorch — the exported graph bakes in the final softmax so
`sentiment.py` never reimplements it, and the graph is self-contained
(no separately-loaded architecture code needed at inference time, which
is what makes the ONNX approach immune to architecture drift). Dynamic
batch dimension (`dynamic_axes` below), fixed seq_len=`MAX_SEQ_LEN` —
`score_sentiment_batch()` calls this with a whole cycle's posts at once,
not one post at a time, so the exported graph needs to support that shape.

In [ ]:
class InferenceWrapper(nn.Module):
    def __init__(self, trained_model):
        super().__init__()
        self.model = trained_model

    def forward(self, input_ids):
        logits = self.model(input_ids)
        return F.softmax(logits, dim=1)


export_model = InferenceWrapper(model).to(DEVICE).eval()
dummy_input = torch.zeros((1, sentiment_model.MAX_SEQ_LEN), dtype=torch.long, device=DEVICE)

torch.onnx.export(
    export_model,
    dummy_input,
    "model.onnx",
    input_names=["input_ids"],
    output_names=["probs"],
    dynamic_axes={"input_ids": {0: "batch"}, "probs": {0: "batch"}},
    opset_version=17,
    dynamo=False,  # forces the stable TorchScript-based exporter; the
    # newer dynamo-based one (PyTorch's new default on some versions)
    # needs the separate `onnxscript` package, which we don't install
)
print("exported model.onnx")

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession("model.onnx", providers=["CPUExecutionProvider"])

with torch.no_grad():
    for text in spot_check_examples:
        ids = sentiment_model.encode(sentiment_model.tokenize(text), vocab)
        ids_np = np.array([ids], dtype=np.int64)

        torch_probs = export_model(torch.tensor(ids_np, device=DEVICE)).cpu().numpy()[0]
        onnx_probs = session.run(None, {"input_ids": ids_np})[0][0]

        assert np.allclose(torch_probs, onnx_probs, atol=1e-4), (text, torch_probs, onnx_probs)

print("ONNX output matches PyTorch output on all spot-check examples")


In [ ]:
# Confirms the dynamic batch dimension actually works end to end, not just
# that the export call didn't raise -- score_sentiment_batch() depends on
# this. Batched call must match the same examples run one at a time above.
batch_ids_np = np.array(
    [sentiment_model.encode(sentiment_model.tokenize(text), vocab) for text in spot_check_examples],
    dtype=np.int64,
)
batch_probs = session.run(None, {"input_ids": batch_ids_np})[0]
assert batch_probs.shape == (len(spot_check_examples), 3)

for text, row in zip(spot_check_examples, batch_probs):
    ids_np = np.array([sentiment_model.encode(sentiment_model.tokenize(text), vocab)], dtype=np.int64)
    single_probs = session.run(None, {"input_ids": ids_np})[0][0]
    assert np.allclose(row, single_probs, atol=1e-4), (text, row, single_probs)

print(f"batched ONNX call (batch={len(spot_check_examples)}) matches per-item single-batch calls")

### Widened parity check

Checked against the full held-out test set, not just a handful of static
spot-check strings — a small sample can hide a real numerical divergence
between the exported graph and the original model. This CNN's export
uses a different toolchain (`torch.onnx.export`, not `skl2onnx`) with no
known bug of that shape, but the same "check at scale" discipline applies
regardless of toolchain.

In [ ]:
parity_sample_size = min(500, len(test_texts))
parity_texts = test_texts[:parity_sample_size]
parity_ids_np = np.array(
    [sentiment_model.encode(sentiment_model.tokenize(t), vocab) for t in parity_texts],
    dtype=np.int64,
)

with torch.no_grad():
    torch_probs_batch = export_model(torch.tensor(parity_ids_np, device=DEVICE)).cpu().numpy()
onnx_probs_batch = session.run(None, {"input_ids": parity_ids_np})[0]

diffs = np.abs(torch_probs_batch - onnx_probs_batch).max(axis=1)
print(f"n = {len(diffs)}")
print(f"max diff: {diffs.max():.6f}")
for p in [50, 90, 99, 99.9]:
    print(f"p{p}: {np.percentile(diffs, p):.6f}")

# Starting tolerance -- same order of magnitude as the spot-check-scale
# assertion above (atol=1e-4), which this export has always cleanly
# passed at n=4. If this fails, DON'T loosen the tolerance to make it
# pass -- that's exactly the mistake category's v5 pass warned against.
# Investigate which examples diverge and why first, the same way the
# skl2onnx sublinear_tf gap was actually diagnosed, not just tolerated.
PARITY_ATOL = 1e-4
assert diffs.max() < PARITY_ATOL, (
    f"ONNX/PyTorch parity check failed at scale: max diff {diffs.max():.6f} >= {PARITY_ATOL}"
)
print(f"ONNX output matches PyTorch output across {len(diffs)} held-out test examples (atol={PARITY_ATOL})")

## Package config.json

Audit metadata only — dataset composition, eval numbers, the commit this
was trained against. Not load-bearing for correctness (the ONNX graph is
self-contained and vocab.json is the only other file inference actually
needs).


In [ ]:
import datetime
import json as jsonlib

VERSION = "v4"  # bump manually for each new published model

model_config = {
    "version": VERSION,
    "architecture_source_commit": SENTIMENT_MODEL_COMMIT,
    "embedding_dim": EMBEDDING_DIM_ACTUAL,
    "filter_sizes": list(sentiment_model.FILTER_SIZES),
    "num_filters": sentiment_model.NUM_FILTERS,
    "dropout": sentiment_model.DROPOUT,
    "max_seq_len": sentiment_model.MAX_SEQ_LEN,
    "vocab_size": len(vocab),
    "embedding_source": EMBEDDING_SOURCE,
    "trained_at": datetime.datetime.utcnow().isoformat() + "Z",
    "sentiment140_max_multiple": SENTIMENT140_MAX_MULTIPLE,
    "dataset_composition": {
        "sentiment140_sampled": len(s140_texts_sampled),
        "tweet_eval_train_val": len(te_train_texts),
        "tweet_sentiment_extraction_train": len(tse_train_texts),
        "go_emotions_single_label": len(ge_texts),
        "dair_emotion_train_val": len(emo_train_texts),
        "combined_total": len(all_texts),
    },
    "best_val_macro_f1": best_val_f1,
}

with open("vocab.json", "w") as f:
    jsonlib.dump(vocab, f)
with open("config.json", "w") as f:
    jsonlib.dump(model_config, f, indent=2)

print(jsonlib.dumps(model_config, indent=2))


## Upload to R2

Cloudflare R2 is S3-API-compatible, so `boto3`'s S3 client works directly
against it. Provide credentials via Colab Secrets / Kaggle Secrets
(preferred) or paste them into the fallback cell below — either way,
**never commit real credentials into this notebook's output**.

Publishing to a specific version directory always happens; **flipping
`latest.json` (what `processing/` actually reads by default) is a
separate, explicitly-gated step** — so a blind "Run All" publishes the
versioned artifacts without silently promoting them to production.


In [ ]:
R2_ACCOUNT_ID = R2_ACCESS_KEY_ID = R2_SECRET_ACCESS_KEY = R2_BUCKET_NAME = None

try:
    from google.colab import userdata

    R2_ACCOUNT_ID = userdata.get("R2_ACCOUNT_ID")
    R2_ACCESS_KEY_ID = userdata.get("R2_ACCESS_KEY_ID")
    R2_SECRET_ACCESS_KEY = userdata.get("R2_SECRET_ACCESS_KEY")
    R2_BUCKET_NAME = userdata.get("R2_BUCKET_NAME")
except Exception as e:
    # covers both "not running in Colab" (ImportError) and Colab Secrets
    # being unreachable in this execution context -- e.g. a non-interactive
    # / background run raises TimeoutException instead, not ImportError,
    # since the one-time permission prompt needs a live UI session
    print(f"Colab secrets unavailable ({type(e).__name__}: {e}), trying Kaggle secrets...")

if not R2_ACCOUNT_ID:
    try:
        from kaggle_secrets import UserSecretsClient

        secrets = UserSecretsClient()
        R2_ACCOUNT_ID = secrets.get_secret("R2_ACCOUNT_ID")
        R2_ACCESS_KEY_ID = secrets.get_secret("R2_ACCESS_KEY_ID")
        R2_SECRET_ACCESS_KEY = secrets.get_secret("R2_SECRET_ACCESS_KEY")
        R2_BUCKET_NAME = secrets.get_secret("R2_BUCKET_NAME")
    except Exception as e:
        print(f"Kaggle secrets unavailable ({type(e).__name__}: {e}), falling back to manual values...")

if not R2_ACCOUNT_ID:
    # Manual fallback -- fill these in locally, never commit real values.
    R2_ACCOUNT_ID = ""
    R2_ACCESS_KEY_ID = ""
    R2_SECRET_ACCESS_KEY = ""
    R2_BUCKET_NAME = ""

assert R2_ACCOUNT_ID and R2_ACCESS_KEY_ID and R2_SECRET_ACCESS_KEY and R2_BUCKET_NAME, (
    "R2 credentials not set -- see the markdown cell above"
)


In [ ]:
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url=f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    region_name="auto",
)

PREFIX = f"sentiment-cnn/{VERSION}"
s3.upload_file("model.onnx", R2_BUCKET_NAME, f"{PREFIX}/model.onnx")
s3.upload_file("vocab.json", R2_BUCKET_NAME, f"{PREFIX}/vocab.json")
s3.upload_file("config.json", R2_BUCKET_NAME, f"{PREFIX}/config.json")
print(f"uploaded to s3://{R2_BUCKET_NAME}/{PREFIX}/")


In [ ]:
# Deliberately gated -- flip to True only after checking the eval numbers
# and spot-check output above. This is what actually makes processing/
# start using this model (it reads sentiment-cnn/latest.json by default).
#
# NOTE (2026-08-12): this only flips latest.json. goodgorithm-models is a
# private R2 bucket, and Colab has no `gh`/repo access, so this cell alone
# cannot make the version publicly downloadable -- it does NOT fulfill the
# "we open-source model weights" commitment on its own. Prefer running
# `cd training && uv run python r2_release.py --model sentiment publish <version>`
# instead (from a machine with an authenticated `gh` CLI) -- it does both
# the latest.json flip and the public GitHub Release in one step. If you
# do use this cell, still run r2_release.py publish afterward; it's
# idempotent on the latest.json flip and will just create the missing
# release. See the release-sentiment-model skill.
PUBLISH_AS_LATEST = False

if PUBLISH_AS_LATEST:
    import io
    import json as jsonlib

    latest_bytes = jsonlib.dumps({"version": VERSION}).encode()
    s3.upload_fileobj(io.BytesIO(latest_bytes), R2_BUCKET_NAME, "sentiment-cnn/latest.json")
    print(f"sentiment-cnn/latest.json now points to {VERSION}")
    print("Reminder: run `r2_release.py publish` too, to create the public GitHub Release.")
else:
    print("PUBLISH_AS_LATEST is False -- latest.json untouched, this version is published but not live")
